# CZU-MHAD: Two-Layer Feature Selection (Per-Modality → Combined)

## Overview
This notebook implements a **two-layer feature selection** pipeline:

**Layer 1 (already done):** Feature selection was run independently on each modality (depth, sensor, skeleton, video). Results are stored in:
- `results_new_depth/`
- `results_new_sensor/`
- `results_new_skeleton/`
- `results_new_video/`

**Layer 2 (this notebook):** 
1. **Combine** the selected features from each modality into a unified feature set
2. **Evaluate** the combined features (Layer 1 result) on HAR
3. **Run a second layer** of feature selection on the combined set

### Combination Strategy:
- **Standard methods** (mutual_info, rfe, lasso): Combine features selected by the *same* method across all 4 modalities. Then run the same FS method again on the combined set.
- **Metaheuristics**: For each fold, find the *best* metaheuristic (highest test accuracy) per modality. Combine those best-selected features. Then run all 14 metaheuristics on that combined set.
- **Baseline**: Combine all baseline (all features) from each modality = full feature set.

All results saved in `results_two_layer/`.

In [11]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
from pathlib import Path
from tqdm import tqdm
from copy import deepcopy
import random
import sys
from scipy import stats
import psutil
import tracemalloc
import gc
import json as json_lib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, classification_report)

# Standard feature selection imports
from sklearn.feature_selection import mutual_info_classif, RFE, SelectKBest
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Import ALL EvoloPy optimizers
sys.path.append('../EvoloPy-master')
from EvoloPy.optimizers import BAT, CS, DE, FFA, GA, GWO, HHO, JAYA, MFO, MVO, PSO, SCA, SSA, WOA

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

# Global device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"PyTorch version: {torch.__version__}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Device: cuda
PyTorch version: 2.9.1+cu128
GPU: NVIDIA GeForce RTX 4090


## Configuration

In [2]:
# Paths
FEATURE_DIR = Path("features_new")

# Layer 1 results directories (already computed)
LAYER1_DIRS = {
    'depth': Path("results_depth_new"),
    'sensor': Path("results_sensor_new"),
    'skeleton': Path("results_skeleton_new"),
}

# Master output directory for two-layer results
RESULTS_ROOT = Path("results_two_layer")
RESULTS_ROOT.mkdir(exist_ok=True)
PLOTS_DIR = RESULTS_ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

# Hyperparameters (same as original)
N_FOLDS = 5  # LOSO
N_EPOCHS = 300
BATCH_SIZE = 32
LEARNING_RATE = 0.001
HIDDEN_DIM = 128

# Feature selection hyperparameters - metaheuristic
N_POPULATION = 20
MAX_ITERATIONS = 30

# For standard methods (target: select ~50% of combined features)
TARGET_FEATURE_PERCENTAGE = 0.5

# All EvoloPy optimizer names and their callable entry points
EVOLOPY_OPTIMIZERS = {
    'BAT': BAT.BAT,
    'CS':  CS.CS,
    'DE':  DE.DE,
    'FFA': FFA.FFA,
    'GA':  GA.GA,
    'GWO': GWO.GWO,
    'HHO': HHO.HHO,
    'JAYA': JAYA.JAYA,
    'MFO': MFO.MFO,
    'MVO': MVO.MVO,
    'PSO': PSO.PSO,
    'SCA': SCA.SCA,
    'SSA': SSA.SSA,
    'WOA': WOA.WOA,
}

STANDARD_METHODS = ['mutual_info', 'rfe', 'lasso']
META_METHODS = [f'meta_{k}' for k in EVOLOPY_OPTIMIZERS.keys()]
# For layer 2, we have: 3 standard methods, and meta_best (which runs all 14 metas)
# Output method names for layer 2:
ALL_METHODS_L2 = ['mutual_info', 'rfe', 'lasso'] + [f'meta_{k}' for k in EVOLOPY_OPTIMIZERS.keys()]

# Create per-method subdirectories
for method in ALL_METHODS_L2:
    (RESULTS_ROOT / method).mkdir(exist_ok=True)
# Also create combined_layer1 directory
(RESULTS_ROOT / "combined_layer1").mkdir(exist_ok=True)

print(f"Configuration:")
print(f"  N_FOLDS: {N_FOLDS}")
print(f"  N_EPOCHS: {N_EPOCHS}")
print(f"  Target feature selection (Layer 2): {TARGET_FEATURE_PERCENTAGE*100:.0f}%")
print(f"  Metaheuristic pop: {N_POPULATION}, iter: {MAX_ITERATIONS}")
print(f"  EvoloPy optimizers: {list(EVOLOPY_OPTIMIZERS.keys())}")
print(f"  Layer 1 dirs: {list(LAYER1_DIRS.keys())}")

# Verify layer 1 directories exist
for mod, d in LAYER1_DIRS.items():
    if d.exists():
        print(f"  ✓ {d} exists")
    else:
        print(f"  ✗ {d} MISSING!")


Configuration:
  N_FOLDS: 5
  N_EPOCHS: 300
  Target feature selection (Layer 2): 50%
  Metaheuristic pop: 20, iter: 30
  EvoloPy optimizers: ['BAT', 'CS', 'DE', 'FFA', 'GA', 'GWO', 'HHO', 'JAYA', 'MFO', 'MVO', 'PSO', 'SCA', 'SSA', 'WOA']
  Layer 1 dirs: ['depth', 'sensor', 'skeleton']
  ✓ results_depth_new exists
  ✓ results_sensor_new exists
  ✓ results_skeleton_new exists


## Load Data

In [4]:
# Load features
X_feat = joblib.load(FEATURE_DIR / "X_feat.pkl")
y = np.load(FEATURE_DIR / "y.npy")
subjects = np.load(FEATURE_DIR / "subjects.npy")
le = joblib.load(FEATURE_DIR / "label_encoder.pkl")
print(f"Loaded {len(X_feat)} samples")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Number of subjects: {len(np.unique(subjects))}")

# Get feature dimensions
first_sample = X_feat[0]
FEATURE_DIMS = {
    'depth': first_sample['depth_feat'].shape[0],
    'sensor': first_sample['sensor_feat'].shape[0],
    'skeleton': first_sample['skeleton_feat'].shape[0]
}
TOTAL_FEATURES = sum(FEATURE_DIMS.values())
print(f"\nFeature dimensions:")
for modality, dim in FEATURE_DIMS.items():
    print(f"  {modality}: {dim}")
print(f"  TOTAL: {TOTAL_FEATURES}")

# Compute modality offsets in the unified feature vector
MODALITY_OFFSETS = {}
offset = 0
for mod, dim in FEATURE_DIMS.items():
    MODALITY_OFFSETS[mod] = (offset, offset + dim)
    offset += dim
print(f"\nModality offsets in unified vector:")
for mod, (s, e) in MODALITY_OFFSETS.items():
    print(f"  {mod}: [{s}, {e})")


Loaded 1165 samples
Number of classes: 22
Number of subjects: 5

Feature dimensions:
  depth: 216
  sensor: 652
  skeleton: 1645
  TOTAL: 2513

Modality offsets in unified vector:
  depth: [0, 216)
  sensor: [216, 868)
  skeleton: [868, 2513)


## Neural Network Model

In [5]:
class MultiModalDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class SimpleNN(nn.Module):
    """MLP for unified feature vector (adaptive to feature subset size)"""
    def __init__(self, input_dim, num_classes):
        super().__init__()
        
        # Adaptive hidden layer sizing based on input dimension
        hidden1 = max(128, min(512, input_dim * 2))
        hidden2 = max(64, min(256, hidden1 // 2))
        
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden2, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(x)

print("Neural network model defined")


Neural network model defined


## Helper Functions

In [6]:
def prepare_unified_features(X_feat_list, feature_mask=None):
    """Concatenate all modality features into unified vector"""
    unified_features = []
    
    for sample in X_feat_list:
        feat_vector = np.concatenate([
            sample['depth_feat'],
            sample['sensor_feat'],
            sample['skeleton_feat']
        ])
        
        if feature_mask is not None:
            feat_vector = feat_vector[feature_mask]
        
        unified_features.append(feat_vector)
    
    return np.array(unified_features)

def calculate_modality_retention(binary_mask, feature_dims):
    """Calculate how many features retained per modality"""
    start_idx = 0
    retention = {}
    
    for modality, dim in feature_dims.items():
        end_idx = start_idx + dim
        modality_mask = binary_mask[start_idx:end_idx]
        num_selected = np.sum(modality_mask)
        percentage = (num_selected / dim) * 100
        
        retention[modality] = {
            'selected': int(num_selected),
            'total': dim,
            'percentage': percentage
        }
        start_idx = end_idx
    
    return retention

def count_model_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_model_size_mb(model):
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 ** 2)

def get_gpu_memory_mb():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / (1024 ** 2)
    return 0.0

def get_dataset_size_mb(X):
    return X.nbytes / (1024 ** 2)

print("Helper functions defined")


Helper functions defined


## Enhanced Evaluation

In [7]:
def train_and_evaluate_full(model, train_loader, val_loader, test_loader, num_epochs, lr, num_classes):
    """Train model and return comprehensive metrics including predictions for confusion matrix"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    gpu_mem_before = get_gpu_memory_mb()
    train_start = time.time()

    train_losses = []
    best_val_acc = -1.0
    best_model_state = None
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for features, labels in train_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        train_losses.append(epoch_loss / len(train_loader))

        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(DEVICE)
                outputs = model(features)
                preds = torch.argmax(outputs, dim=1)
                val_correct += (preds.cpu() == labels).sum().item()
                val_total += labels.size(0)

        epoch_val_acc = val_correct / val_total

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_model_state = deepcopy(model.state_dict())
    
    train_time = time.time() - train_start
    gpu_mem_after = get_gpu_memory_mb()

    model.load_state_dict(best_model_state)
    
    model.eval()
    with torch.no_grad():
        val_preds, val_true = [], []
        for features, labels in val_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.numpy())
        
        test_preds, test_true = [], []
        for features, labels in test_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            test_preds.extend(preds)
            test_true.extend(labels.numpy())
    
    val_acc = accuracy_score(val_true, val_preds)
    test_acc = accuracy_score(test_true, test_preds)
    
    metrics = {
        'val_acc': val_acc,
        'test_acc': test_acc,
        'test_f1_macro': f1_score(test_true, test_preds, average='macro', zero_division=0),
        'test_f1_weighted': f1_score(test_true, test_preds, average='weighted', zero_division=0),
        'test_precision_macro': precision_score(test_true, test_preds, average='macro', zero_division=0),
        'test_recall_macro': recall_score(test_true, test_preds, average='macro', zero_division=0),
        'test_preds': np.array(test_preds),
        'test_true': np.array(test_true),
        'val_preds': np.array(val_preds),
        'val_true': np.array(val_true),
        'train_time_sec': train_time,
        'gpu_mem_before_mb': gpu_mem_before,
        'gpu_mem_after_mb': gpu_mem_after,
        'gpu_mem_peak_mb': torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0,
        'model_params': count_model_parameters(model),
        'model_size_mb': get_model_size_mb(model),
        'train_losses': train_losses,
    }
    return metrics

print("Enhanced evaluation function defined")


Enhanced evaluation function defined


## Load Layer 1 Results (Pre-computed Per-Modality Feature Selection)

Load the feature masks from each modality's per-fold results.

In [8]:
def load_layer1_masks():
    """
    Load all Layer 1 feature masks from per-modality results directories.
    
    Returns:
        layer1_masks: dict[modality][method][fold_idx] = binary_mask (numpy bool array)
        layer1_results: dict[modality][method][fold_idx] = json_results (dict)
    """
    layer1_masks = {}
    layer1_results = {}
    
    all_methods_l1 = ['mutual_info', 'rfe', 'lasso'] + [f'meta_{k}' for k in EVOLOPY_OPTIMIZERS.keys()]
    
    for modality, results_dir in LAYER1_DIRS.items():
        layer1_masks[modality] = {}
        layer1_results[modality] = {}
        
        for method in all_methods_l1:
            layer1_masks[modality][method] = {}
            layer1_results[modality][method] = {}
            
            method_dir = results_dir / method
            if not method_dir.exists():
                print(f"  WARNING: {method_dir} does not exist!")
                continue
            
            for fold_idx in range(N_FOLDS):
                mask_path = method_dir / f"fold_{fold_idx+1}_mask.npy"
                json_path = method_dir / f"fold_{fold_idx+1}.json"
                
                if mask_path.exists():
                    mask = np.load(mask_path)
                    layer1_masks[modality][method][fold_idx] = mask
                else:
                    print(f"  WARNING: Missing mask: {mask_path}")
                
                if json_path.exists():
                    with open(json_path) as f:
                        layer1_results[modality][method][fold_idx] = json_lib.load(f)
    
    return layer1_masks, layer1_results

layer1_masks, layer1_results = load_layer1_masks()

# Summary
print("\n" + "="*80)
print("LAYER 1 MASKS LOADED")
print("="*80)
modalities = list(LAYER1_DIRS.keys())
all_methods_l1 = ['mutual_info', 'rfe', 'lasso'] + [f'meta_{k}' for k in EVOLOPY_OPTIMIZERS.keys()]

for modality in modalities:
    n_loaded = sum(
        len(layer1_masks[modality].get(m, {})) 
        for m in all_methods_l1
    )
    print(f"  {modality}: {n_loaded} fold-masks loaded across {len(all_methods_l1)} methods")



LAYER 1 MASKS LOADED
  depth: 85 fold-masks loaded across 17 methods
  sensor: 85 fold-masks loaded across 17 methods
  skeleton: 85 fold-masks loaded across 17 methods


## Combine Layer 1 Selected Features Into Unified Masks

### Strategy:
- **Standard methods** (mutual_info, rfe, lasso): For each fold, take the selected features from the same method in each modality and combine them (union into unified feature space).
- **Metaheuristics**: For each fold, find the best metaheuristic per modality (highest test accuracy), then combine those best-selected features.
- **Baseline**: All features from all modalities (full feature set).

The combined mask is over the full unified feature space (depth + sensor + skeleton + video).

In [9]:
def build_combined_mask_standard(method, fold_idx):
    """
    For a standard method, combine selected features from the same method 
    across all 3 modalities into a unified mask over the full feature space.
    
    Each modality mask is a boolean array over that modality's features only.
    We map them to their correct positions in the unified vector.
    """
    unified_mask = np.zeros(TOTAL_FEATURES, dtype=bool)
    
    for modality in modalities:
        if method not in layer1_masks[modality]:
            continue
        if fold_idx not in layer1_masks[modality][method]:
            continue
        
        mod_mask = layer1_masks[modality][method][fold_idx]
        start, end = MODALITY_OFFSETS[modality]
        mod_dim = end - start
        
        # The modality mask length should match the modality dimension
        if len(mod_mask) == mod_dim:
            unified_mask[start:end] = mod_mask
        else:
            print(f"  WARNING: {modality}/{method}/fold_{fold_idx+1} mask length "
                  f"{len(mod_mask)} != expected {mod_dim}")
    
    return unified_mask


def find_best_meta_per_modality(fold_idx):
    """
    For a given fold, find which metaheuristic gave the highest test accuracy
    in each modality. Returns dict[modality] = (best_method_name, test_acc).
    """
    best_per_mod = {}
    
    for modality in modalities:
        best_method = None
        best_acc = -1.0
        
        for method in META_METHODS:
            if method not in layer1_results[modality]:
                continue
            if fold_idx not in layer1_results[modality][method]:
                continue
            
            result = layer1_results[modality][method][fold_idx]
            test_acc = result.get('test_acc', 0)
            
            if test_acc > best_acc:
                best_acc = test_acc
                best_method = method
        
        best_per_mod[modality] = (best_method, best_acc)
    
    return best_per_mod


def build_combined_mask_best_meta(fold_idx):
    """
    For metaheuristics: find best meta per modality for this fold,
    then combine their selected features into a unified mask.
    """
    unified_mask = np.zeros(TOTAL_FEATURES, dtype=bool)
    best_per_mod = find_best_meta_per_modality(fold_idx)
    
    for modality in modalities:
        best_method, best_acc = best_per_mod[modality]
        if best_method is None:
            print(f"  WARNING: No meta results for {modality} fold {fold_idx+1}")
            continue
        
        mod_mask = layer1_masks[modality][best_method][fold_idx]
        start, end = MODALITY_OFFSETS[modality]
        mod_dim = end - start
        
        if len(mod_mask) == mod_dim:
            unified_mask[start:end] = mod_mask
        else:
            print(f"  WARNING: {modality}/{best_method}/fold_{fold_idx+1} mask length "
                  f"{len(mod_mask)} != expected {mod_dim}")
    
    return unified_mask, best_per_mod


# Build all combined masks
combined_masks = {}  # method -> {fold_idx: mask}

# Standard methods
for method in STANDARD_METHODS:
    combined_masks[method] = {}
    for fold_idx in range(N_FOLDS):
        combined_masks[method][fold_idx] = build_combined_mask_standard(method, fold_idx)

# Best metaheuristic combination
combined_masks['meta_best'] = {}
meta_best_info = {}  # fold_idx -> best_per_mod
for fold_idx in range(N_FOLDS):
    mask, best_per_mod = build_combined_mask_best_meta(fold_idx)
    combined_masks['meta_best'][fold_idx] = mask
    meta_best_info[fold_idx] = best_per_mod

# Print summary
print("="*80)
print("COMBINED LAYER 1 MASKS SUMMARY")
print("="*80)

for method in STANDARD_METHODS + ['meta_best']:
    avg_selected = np.mean([np.sum(combined_masks[method][f]) for f in range(N_FOLDS)])
    print(f"  {method:15s}: avg {avg_selected:.0f}/{TOTAL_FEATURES} features selected "
          f"({avg_selected/TOTAL_FEATURES*100:.1f}%)")

print("\nBest metaheuristic per modality per fold:")
for fold_idx in range(N_FOLDS):
    info = meta_best_info[fold_idx]
    parts = [f"{mod}={info[mod][0]}({info[mod][1]*100:.1f}%)" for mod in modalities]
    print(f"  Fold {fold_idx+1}: {', '.join(parts)}")


COMBINED LAYER 1 MASKS SUMMARY
  mutual_info    : avg 1256/2513 features selected (50.0%)
  rfe            : avg 1256/2513 features selected (50.0%)
  lasso          : avg 1256/2513 features selected (50.0%)
  meta_best      : avg 1222/2513 features selected (48.6%)

Best metaheuristic per modality per fold:
  Fold 1: depth=meta_BAT(44.1%), sensor=meta_BAT(86.0%), skeleton=meta_FFA(97.2%)
  Fold 2: depth=meta_DE(38.6%), sensor=meta_GWO(90.9%), skeleton=meta_JAYA(89.2%)
  Fold 3: depth=meta_CS(35.5%), sensor=meta_PSO(85.5%), skeleton=meta_MFO(96.4%)
  Fold 4: depth=meta_GA(25.0%), sensor=meta_FFA(79.5%), skeleton=meta_GA(95.9%)
  Fold 5: depth=meta_GWO(32.8%), sensor=meta_GA(87.9%), skeleton=meta_MFO(97.0%)


## Evaluate Combined Layer 1 Features (Before Second Layer FS)

Train and evaluate the neural network on the combined features from Layer 1,
**without** any additional feature selection. This shows the HAR performance
of simply combining per-modality selected features.

In [10]:
def evaluate_combined_layer1():
    """
    Evaluate combined Layer 1 features (no second-layer FS yet).
    This gives us the baseline performance of the combined per-modality selections.
    """
    print("="*80)
    print("EVALUATING COMBINED LAYER 1 FEATURES (before Layer 2 FS)")
    print("="*80)
    
    X_unified = prepare_unified_features(X_feat)
    num_classes = len(np.unique(y))
    
    gkf = GroupKFold(n_splits=N_FOLDS)
    
    layer1_eval_results = {}  # method -> {fold_idx: metrics}
    
    methods_to_eval = STANDARD_METHODS + ['meta_best']
    
    for method in methods_to_eval:
        layer1_eval_results[method] = {}
    
    for fold_idx, (train_val_idx, test_idx) in enumerate(gkf.split(X_unified, y, groups=subjects)):
        print(f"\n{'#'*60}")
        print(f"# FOLD {fold_idx + 1}/{N_FOLDS}")
        print(f"{'#'*60}")
        
        # Split data
        X_train_val = X_unified[train_val_idx]
        y_train_val = y[train_val_idx]
        X_test = X_unified[test_idx]
        y_test = y[test_idx]
        
        # Further split train_val into train and val (same as original)
        np.random.seed(42)  # ensure same split as original
        n_val = len(train_val_idx) // 5
        val_indices = np.random.choice(len(train_val_idx), n_val, replace=False)
        train_indices = np.array([i for i in range(len(train_val_idx)) if i not in val_indices])
        
        X_train = X_train_val[train_indices]
        y_train = y_train_val[train_indices]
        X_val = X_train_val[val_indices]
        y_val = y_train_val[val_indices]
        
        # Normalize
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)
        
        orig_dataset_size_mb = get_dataset_size_mb(X_train) + get_dataset_size_mb(X_val) + get_dataset_size_mb(X_test)
        
        for method in methods_to_eval:
            print(f"\n  --- Combined L1: {method.upper()} ---")
            
            feature_mask = combined_masks[method][fold_idx]
            
            X_train_sel = X_train[:, feature_mask]
            X_val_sel = X_val[:, feature_mask]
            X_test_sel = X_test[:, feature_mask]
            
            opt_dataset_size_mb = get_dataset_size_mb(X_train_sel) + get_dataset_size_mb(X_val_sel) + get_dataset_size_mb(X_test_sel)
            
            model = SimpleNN(X_train_sel.shape[1], num_classes).to(DEVICE)
            
            train_dataset = MultiModalDataset(X_train_sel, y_train)
            val_dataset = MultiModalDataset(X_val_sel, y_val)
            test_dataset = MultiModalDataset(X_test_sel, y_test)
            
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
            test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
            
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            
            eval_metrics = train_and_evaluate_full(
                model, train_loader, val_loader, test_loader, N_EPOCHS, LEARNING_RATE, num_classes
            )
            
            modality_ret = calculate_modality_retention(feature_mask, FEATURE_DIMS)
            
            fold_result = {
                'fold': fold_idx,
                'method': f'combined_L1_{method}',
                'val_acc': eval_metrics['val_acc'],
                'test_acc': eval_metrics['test_acc'],
                'test_f1_macro': eval_metrics['test_f1_macro'],
                'test_f1_weighted': eval_metrics['test_f1_weighted'],
                'test_precision_macro': eval_metrics['test_precision_macro'],
                'test_recall_macro': eval_metrics['test_recall_macro'],
                'num_features_selected': int(np.sum(feature_mask)),
                'num_features_total': TOTAL_FEATURES,
                'feature_retention_pct': float(np.sum(feature_mask) / TOTAL_FEATURES * 100),
                'modality_retention': modality_ret,
                'feature_mask': feature_mask,
                'train_time_sec': eval_metrics['train_time_sec'],
                'total_time_sec': eval_metrics['train_time_sec'],
                'model_params': eval_metrics['model_params'],
                'model_size_mb': eval_metrics['model_size_mb'],
                'original_dataset_size_mb': orig_dataset_size_mb,
                'optimized_dataset_size_mb': opt_dataset_size_mb,
                'dataset_reduction_pct': float((1 - opt_dataset_size_mb / orig_dataset_size_mb) * 100) if orig_dataset_size_mb > 0 else 0,
                'gpu_mem_peak_mb': eval_metrics['gpu_mem_peak_mb'],
            }
            
            layer1_eval_results[method][fold_idx] = fold_result
            
            # Save
            save_dir = RESULTS_ROOT / "combined_layer1"
            fold_save = {k: v for k, v in fold_result.items() if k != 'feature_mask'}
            fold_save_clean = {}
            for k, v in fold_save.items():
                if isinstance(v, np.ndarray):
                    fold_save_clean[k] = v.tolist()
                elif isinstance(v, (np.floating, np.integer)):
                    fold_save_clean[k] = float(v)
                elif isinstance(v, dict):
                    fold_save_clean[k] = {}
                    for kk, vv in v.items():
                        if isinstance(vv, dict):
                            fold_save_clean[k][kk] = {kkk: float(vvv) if isinstance(vvv, (np.floating, np.integer)) else vvv for kkk, vvv in vv.items()}
                        else:
                            fold_save_clean[k][kk] = float(vv) if isinstance(vv, (np.floating, np.integer)) else vv
                else:
                    fold_save_clean[k] = v
            
            with open(save_dir / f"combined_L1_{method}_fold_{fold_idx+1}.json", 'w') as f:
                json_lib.dump(fold_save_clean, f, indent=2, default=str)
            
            np.save(save_dir / f"combined_L1_{method}_fold_{fold_idx+1}_mask.npy", feature_mask)
            
            print(f"    Test Acc: {eval_metrics['test_acc']*100:.2f}%, "
                  f"F1: {eval_metrics['test_f1_macro']*100:.2f}%, "
                  f"Features: {np.sum(feature_mask)}/{TOTAL_FEATURES} "
                  f"({np.sum(feature_mask)/TOTAL_FEATURES*100:.1f}%)")
            
            del model, train_dataset, val_dataset, test_dataset
            del train_loader, val_loader, test_loader
            torch.cuda.empty_cache()
            gc.collect()
    
    return layer1_eval_results

layer1_eval_results = evaluate_combined_layer1()


EVALUATING COMBINED LAYER 1 FEATURES (before Layer 2 FS)

############################################################
# FOLD 1/5
############################################################

  --- Combined L1: MUTUAL_INFO ---
    Test Acc: 99.30%, F1: 99.30%, Features: 1256/2513 (50.0%)

  --- Combined L1: RFE ---
    Test Acc: 99.65%, F1: 99.65%, Features: 1256/2513 (50.0%)

  --- Combined L1: LASSO ---
    Test Acc: 94.06%, F1: 93.78%, Features: 1256/2513 (50.0%)

  --- Combined L1: META_BEST ---
    Test Acc: 94.76%, F1: 94.77%, Features: 1285/2513 (51.1%)

############################################################
# FOLD 2/5
############################################################

  --- Combined L1: MUTUAL_INFO ---
    Test Acc: 90.87%, F1: 91.17%, Features: 1256/2513 (50.0%)

  --- Combined L1: RFE ---
    Test Acc: 94.19%, F1: 94.39%, Features: 1256/2513 (50.0%)

  --- Combined L1: LASSO ---
    Test Acc: 91.70%, F1: 91.85%, Features: 1256/2513 (50.0%)

  --- Combined L1:

## Layer 1 Combined Results Summary

In [12]:
print("\n" + "="*80)
print("COMBINED LAYER 1 RESULTS SUMMARY")
print("="*80)

methods_to_eval = STANDARD_METHODS + ['meta_best']

rows = []
for method in methods_to_eval:
    accs = []
    f1s = []
    feats = []
    for fold_idx in range(N_FOLDS):
        if fold_idx in layer1_eval_results[method]:
            r = layer1_eval_results[method][fold_idx]
            accs.append(r['test_acc'] * 100)
            f1s.append(r['test_f1_macro'] * 100)
            feats.append(r['num_features_selected'])
    
    if accs:
        rows.append({
            'Method': f'Combined_L1_{method}',
            'Mean Test Acc (%)': np.mean(accs),
            'Std Test Acc (%)': np.std(accs),
            'Mean F1 Macro (%)': np.mean(f1s),
            'Mean Features': np.mean(feats),
            'Mean Retention (%)': np.mean(feats) / TOTAL_FEATURES * 100,
        })

df_l1_summary = pd.DataFrame(rows).round(2)
print(df_l1_summary.to_string(index=False))

# Save
df_l1_summary.to_csv(RESULTS_ROOT / "combined_layer1_summary.csv", index=False)
print(f"\nSaved: combined_layer1_summary.csv")



COMBINED LAYER 1 RESULTS SUMMARY
                 Method  Mean Test Acc (%)  Std Test Acc (%)  Mean F1 Macro (%)  Mean Features  Mean Retention (%)
Combined_L1_mutual_info              95.93              3.95              95.66         1256.0               49.98
        Combined_L1_rfe              96.00              3.72              95.73         1256.0               49.98
      Combined_L1_lasso              93.82              1.92              93.70         1256.0               49.98
  Combined_L1_meta_best              94.65              2.31              94.45         1221.8               48.62

Saved: combined_layer1_summary.csv


## Feature Selection Methods for Layer 2

These are the same FS methods as the original notebook, but now they operate
on the **combined Layer 1 features** (a subset of the full feature space).

In [13]:
def train_model_quick(model, train_loader, val_loader, epochs, lr, device):
    """Quick training for fitness evaluation"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float('inf')
    best_val_acc = 0.0
    
    for epoch in range(epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                val_loss += criterion(outputs, labels).item()
                val_correct += (torch.argmax(outputs, 1) == labels).sum().item()
                val_total += labels.size(0)
        
        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        if val_loss < best_val_loss:
            best_val_loss, best_val_acc = val_loss, val_acc
    
    return best_val_loss, best_val_acc


def create_fitness_function_evolopy_l2(X_train, y_train, X_val, y_val, num_classes, n_features_l2):
    """Create fitness function for EvoloPy Layer 2"""
    eval_count = [0]
    
    def fitness_function(binary_mask):
        try:
            if binary_mask.dtype != bool:
                binary_mask = binary_mask > 0.5

            num_selected = np.sum(binary_mask)
            if num_selected == 0:
                return 1.0
            
            X_tr_sel = X_train[:, binary_mask]
            X_val_sel = X_val[:, binary_mask]
            
            train_dataset = MultiModalDataset(X_tr_sel, y_train)
            val_dataset = MultiModalDataset(X_val_sel, y_val)
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
            
            model = SimpleNN(X_tr_sel.shape[1], num_classes).to(DEVICE)
            
            val_loss, val_acc = train_model_quick(
                model, train_loader, val_loader,
                epochs=30, lr=1e-3, device=DEVICE
            )
            
            del model, train_dataset, val_dataset, train_loader, val_loader
            torch.cuda.empty_cache()
            
            eval_count[0] += 1
            
            alpha = 0.95
            beta = 0.05
            feature_ratio = num_selected / len(binary_mask)

            fitness = alpha * (1.0 - val_acc) + beta * feature_ratio
            return fitness
            
        except Exception as e:
            print(f"Error in fitness: {e}")
            return 1.0
        
    return fitness_function


def s_transfer(x):
    return 1 / (1 + np.exp(-10 * (x - 0.5)))


def run_evolopy_optimizer_l2(optimizer_name, optimizer_func, X_train, y_train, X_val, y_val, num_classes, n_features_l2):
    """Run any EvoloPy optimizer on Layer 2 features"""
    print(f"    Running EvoloPy {optimizer_name} (Layer 2)...")
    print(f"      Features: {n_features_l2}, Pop: {N_POPULATION}, Iter: {MAX_ITERATIONS}")
    
    fitness_func = create_fitness_function_evolopy_l2(X_train, y_train, X_val, y_val, num_classes, n_features_l2)
    
    start = time.time()
    solution = optimizer_func(fitness_func, 0, 1, n_features_l2, N_POPULATION, MAX_ITERATIONS)
    exec_time = time.time() - start
    
    binary_mask = np.random.rand(n_features_l2) < s_transfer(solution.bestIndividual)
    
    results = {
        'mask': binary_mask,
        'convergence': solution.convergence.tolist() if hasattr(solution.convergence, 'tolist') else list(solution.convergence),
        'best_fitness': float(solution.convergence[-1]) if len(solution.convergence) > 0 else float('inf'),
        'execution_time': exec_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': n_features_l2,
        'method': f'meta_{optimizer_name}'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Fitness: {results['best_fitness']:.4f}, Time: {exec_time:.1f}s")
    
    return results


def run_mutual_info_fs_l2(X_train, y_train, X_val, y_val, num_classes, n_features_l2):
    """Run Mutual Information feature selection on Layer 2"""
    print("    Running Mutual Information (Layer 2)...")
    
    start_time = time.time()
    mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
    
    k = max(1, int(n_features_l2 * TARGET_FEATURE_PERCENTAGE))
    top_k_indices = np.argsort(mi_scores)[::-1][:k]
    
    binary_mask = np.zeros(n_features_l2, dtype=bool)
    binary_mask[top_k_indices] = True
    
    execution_time = time.time() - start_time
    
    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': n_features_l2,
        'method': 'mutual_info'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")
    
    return results


def run_rfe_fs_l2(X_train, y_train, X_val, y_val, num_classes, n_features_l2):
    """Run RFE feature selection on Layer 2"""
    print("    Running RFE (Layer 2)...")
    
    start_time = time.time()
    estimator = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    
    k = max(1, int(n_features_l2 * TARGET_FEATURE_PERCENTAGE))
    step = max(1, min(50, n_features_l2 // 10))
    selector = RFE(estimator, n_features_to_select=k, step=step)
    selector.fit(X_train, y_train)
    
    binary_mask = selector.support_
    execution_time = time.time() - start_time
    
    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': n_features_l2,
        'method': 'rfe'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")
    
    return results


def run_lasso_fs_l2(X_train, y_train, X_val, y_val, num_classes, n_features_l2):
    """Run LASSO feature selection on Layer 2"""
    print("    Running LASSO (Layer 2)...")
    start_time = time.time()

    lasso = LogisticRegression(
        penalty='l1',
        C=0.01,
        solver='liblinear',
        random_state=42,
        max_iter=1000
    )
    lasso.fit(X_train, y_train)

    coef_abs = np.abs(lasso.coef_).sum(axis=0)

    target_count = max(1, int(n_features_l2 * TARGET_FEATURE_PERCENTAGE))
    top_indices = np.argsort(coef_abs)[::-1][:target_count]
    binary_mask = np.zeros(n_features_l2, dtype=bool)
    binary_mask[top_indices] = True

    execution_time = time.time() - start_time

    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': n_features_l2,
        'method': 'lasso'
    }

    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")

    return results

print("Layer 2 feature selection methods defined")


Layer 2 feature selection methods defined


## Main Layer 2 Experiment Runner

### Process for each fold:
1. **Standard methods** (mutual_info, rfe, lasso): 
   - Take combined L1 mask → extract those features from unified data
   - Run the *same* FS method on this subset
   - The L2 mask is relative to the L1-selected features; map back to unified space
   
2. **Metaheuristics**:
   - Take combined best-meta L1 mask → extract those features
   - Run ALL 14 metaheuristics on this subset

In [14]:
def save_fold_result(fold_result, method_name, fold_idx):
    """Save a fold result to disk"""
    fold_dir = RESULTS_ROOT / method_name
    fold_dir.mkdir(exist_ok=True)
    
    fold_save = {k: v for k, v in fold_result.items() if k not in ['feature_mask']}
    fold_save_clean = {}
    for k, v in fold_save.items():
        if isinstance(v, np.ndarray):
            fold_save_clean[k] = v.tolist()
        elif isinstance(v, (np.floating, np.integer)):
            fold_save_clean[k] = float(v)
        elif isinstance(v, dict):
            fold_save_clean[k] = {}
            for kk, vv in v.items():
                if isinstance(vv, dict):
                    fold_save_clean[k][kk] = {kkk: float(vvv) if isinstance(vvv, (np.floating, np.integer)) else vvv for kkk, vvv in vv.items()}
                else:
                    fold_save_clean[k][kk] = float(vv) if isinstance(vv, (np.floating, np.integer)) else vv
            
        else:
            fold_save_clean[k] = v
    
    with open(fold_dir / f"fold_{fold_idx+1}.json", 'w') as f:
        json_lib.dump(fold_save_clean, f, indent=2, default=str)
    
    np.save(fold_dir / f"fold_{fold_idx+1}_mask.npy", fold_result['feature_mask'])


def run_layer2_experiments():
    """
    Run Layer 2 feature selection on combined Layer 1 features.
    """
    print("="*80)
    print("STARTING LAYER 2 FEATURE SELECTION EXPERIMENTS")
    print("="*80)
    
    X_unified = prepare_unified_features(X_feat)
    num_classes = len(np.unique(y))
    
    master_results = {m: {} for m in ALL_METHODS_L2}
    
    gkf = GroupKFold(n_splits=N_FOLDS)
    
    for fold_idx, (train_val_idx, test_idx) in enumerate(gkf.split(X_unified, y, groups=subjects)):
        print(f"\n{'#'*80}")
        print(f"# FOLD {fold_idx + 1}/{N_FOLDS}")
        print(f"{'#'*80}")
        
        # Split data (same as original)
        X_train_val = X_unified[train_val_idx]
        y_train_val = y[train_val_idx]
        X_test = X_unified[test_idx]
        y_test = y[test_idx]
        
        np.random.seed(42)
        n_val = len(train_val_idx) // 5
        val_indices = np.random.choice(len(train_val_idx), n_val, replace=False)
        train_indices = np.array([i for i in range(len(train_val_idx)) if i not in val_indices])
        
        X_train = X_train_val[train_indices]
        y_train = y_train_val[train_indices]
        X_val = X_train_val[val_indices]
        y_val = y_train_val[val_indices]
        
        print(f"  Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
        
        # Normalize full unified features
        scaler = StandardScaler()
        X_train_norm = scaler.fit_transform(X_train)
        X_val_norm = scaler.transform(X_val)
        X_test_norm = scaler.transform(X_test)
        
        orig_dataset_size_mb = get_dataset_size_mb(X_train_norm) + get_dataset_size_mb(X_val_norm) + get_dataset_size_mb(X_test_norm)
        
        # =================================================================
        # STANDARD METHODS: mutual_info, rfe, lasso
        # =================================================================
        for std_method in STANDARD_METHODS:
            print(f"\n  --- Layer 2: {std_method.upper()} ---")
            
            # Get combined L1 mask for this method
            l1_mask = combined_masks[std_method][fold_idx]
            l1_indices = np.where(l1_mask)[0]  # indices in unified space
            n_l1_features = len(l1_indices)
            
            print(f"    L1 combined features: {n_l1_features}/{TOTAL_FEATURES}")
            
            # Extract L1-selected features
            X_train_l1 = X_train_norm[:, l1_mask]
            X_val_l1 = X_val_norm[:, l1_mask]
            X_test_l1 = X_test_norm[:, l1_mask]
            
            fs_start_time = time.time()
            
            # Run Layer 2 FS on the L1-selected features
            if std_method == 'mutual_info':
                fs_results = run_mutual_info_fs_l2(X_train_l1, y_train, X_val_l1, y_val, num_classes, n_l1_features)
            elif std_method == 'rfe':
                fs_results = run_rfe_fs_l2(X_train_l1, y_train, X_val_l1, y_val, num_classes, n_l1_features)
            elif std_method == 'lasso':
                fs_results = run_lasso_fs_l2(X_train_l1, y_train, X_val_l1, y_val, num_classes, n_l1_features)
            
            fs_time = time.time() - fs_start_time
            
            # L2 mask is relative to L1 features; map back to unified space
            l2_mask_relative = fs_results['mask']
            
            # Final unified mask
            final_mask = np.zeros(TOTAL_FEATURES, dtype=bool)
            final_mask[l1_indices[l2_mask_relative]] = True
            
            # Apply final mask
            X_train_sel = X_train_norm[:, final_mask]
            X_val_sel = X_val_norm[:, final_mask]
            X_test_sel = X_test_norm[:, final_mask]
            
            opt_dataset_size_mb = get_dataset_size_mb(X_train_sel) + get_dataset_size_mb(X_val_sel) + get_dataset_size_mb(X_test_sel)
            
            model = SimpleNN(X_train_sel.shape[1], num_classes).to(DEVICE)
            train_dataset = MultiModalDataset(X_train_sel, y_train)
            val_dataset = MultiModalDataset(X_val_sel, y_val)
            test_dataset = MultiModalDataset(X_test_sel, y_test)
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
            test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
            
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            
            eval_metrics = train_and_evaluate_full(model, train_loader, val_loader, test_loader, N_EPOCHS, LEARNING_RATE, num_classes)
            modality_ret = calculate_modality_retention(final_mask, FEATURE_DIMS)
            
            fold_result = {
                'fold': fold_idx, 'method': std_method,
                'l1_method': std_method, 'l1_features': n_l1_features,
                'val_acc': eval_metrics['val_acc'], 'test_acc': eval_metrics['test_acc'],
                'test_f1_macro': eval_metrics['test_f1_macro'], 'test_f1_weighted': eval_metrics['test_f1_weighted'],
                'test_precision_macro': eval_metrics['test_precision_macro'], 'test_recall_macro': eval_metrics['test_recall_macro'],
                'num_features_selected': int(np.sum(final_mask)),
                'num_features_total': TOTAL_FEATURES,
                'feature_retention_pct': float(np.sum(final_mask) / TOTAL_FEATURES * 100),
                'modality_retention': modality_ret, 'feature_mask': final_mask,
                'fs_execution_time': fs_results.get('execution_time', 0),
                'train_time_sec': eval_metrics['train_time_sec'],
                'total_time_sec': fs_time + eval_metrics['train_time_sec'],
                'model_params': eval_metrics['model_params'], 'model_size_mb': eval_metrics['model_size_mb'],
                'original_dataset_size_mb': orig_dataset_size_mb, 'optimized_dataset_size_mb': opt_dataset_size_mb,
                'dataset_reduction_pct': float((1 - opt_dataset_size_mb / orig_dataset_size_mb) * 100) if orig_dataset_size_mb > 0 else 0,
                'gpu_mem_peak_mb': eval_metrics['gpu_mem_peak_mb'],
                'convergence': None, 'best_fitness': None,
            }
            
            master_results[std_method][fold_idx] = fold_result
            save_fold_result(fold_result, std_method, fold_idx)
            
            print(f"    L2 Test Acc: {eval_metrics['test_acc']*100:.2f}%, "
                  f"F1: {eval_metrics['test_f1_macro']*100:.2f}%, "
                  f"Features: {np.sum(final_mask)}/{TOTAL_FEATURES} "
                  f"({np.sum(final_mask)/TOTAL_FEATURES*100:.1f}%)")
            
            del model, train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader
            torch.cuda.empty_cache(); gc.collect()
        
        # =================================================================
        # METAHEURISTICS: Run all 14 on combined best-meta L1 features
        # =================================================================
        l1_mask_meta = combined_masks['meta_best'][fold_idx]
        l1_indices_meta = np.where(l1_mask_meta)[0]
        n_l1_meta_features = len(l1_indices_meta)
        
        print(f"\n  --- Layer 2: METAHEURISTICS (on best-meta combined L1) ---")
        print(f"    L1 combined best-meta features: {n_l1_meta_features}/{TOTAL_FEATURES}")
        best_info = meta_best_info[fold_idx]
        for mod in modalities:
            print(f"      {mod}: {best_info[mod][0]} (acc={best_info[mod][1]*100:.1f}%)")
        
        X_train_l1_meta = X_train_norm[:, l1_mask_meta]
        X_val_l1_meta = X_val_norm[:, l1_mask_meta]
        X_test_l1_meta = X_test_norm[:, l1_mask_meta]
        
        for opt_name, opt_func in EVOLOPY_OPTIMIZERS.items():
            method_name = f'meta_{opt_name}'
            print(f"\n  --- Layer 2: {method_name.upper()} ---")
            
            fs_start_time = time.time()
            fs_results = run_evolopy_optimizer_l2(
                opt_name, opt_func, 
                X_train_l1_meta, y_train, X_val_l1_meta, y_val, 
                num_classes, n_l1_meta_features
            )
            fs_time = time.time() - fs_start_time
            
            l2_mask_relative = fs_results['mask']
            
            final_mask = np.zeros(TOTAL_FEATURES, dtype=bool)
            final_mask[l1_indices_meta[l2_mask_relative]] = True
            
            X_train_sel = X_train_norm[:, final_mask]
            X_val_sel = X_val_norm[:, final_mask]
            X_test_sel = X_test_norm[:, final_mask]
            
            opt_dataset_size_mb = get_dataset_size_mb(X_train_sel) + get_dataset_size_mb(X_val_sel) + get_dataset_size_mb(X_test_sel)
            
            model = SimpleNN(X_train_sel.shape[1], num_classes).to(DEVICE)
            train_dataset = MultiModalDataset(X_train_sel, y_train)
            val_dataset = MultiModalDataset(X_val_sel, y_val)
            test_dataset = MultiModalDataset(X_test_sel, y_test)
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
            test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
            
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            
            eval_metrics = train_and_evaluate_full(model, train_loader, val_loader, test_loader, N_EPOCHS, LEARNING_RATE, num_classes)
            modality_ret = calculate_modality_retention(final_mask, FEATURE_DIMS)
            
            fold_result = {
                'fold': fold_idx, 'method': method_name,
                'l1_method': 'meta_best', 'l1_features': n_l1_meta_features,
                'l1_best_per_modality': {mod: {'method': best_info[mod][0], 'acc': best_info[mod][1]} for mod in modalities},
                'val_acc': eval_metrics['val_acc'], 'test_acc': eval_metrics['test_acc'],
                'test_f1_macro': eval_metrics['test_f1_macro'], 'test_f1_weighted': eval_metrics['test_f1_weighted'],
                'test_precision_macro': eval_metrics['test_precision_macro'], 'test_recall_macro': eval_metrics['test_recall_macro'],
                'num_features_selected': int(np.sum(final_mask)),
                'num_features_total': TOTAL_FEATURES,
                'feature_retention_pct': float(np.sum(final_mask) / TOTAL_FEATURES * 100),
                'modality_retention': modality_ret, 'feature_mask': final_mask,
                'fs_execution_time': fs_results.get('execution_time', 0),
                'train_time_sec': eval_metrics['train_time_sec'],
                'total_time_sec': fs_time + eval_metrics['train_time_sec'],
                'model_params': eval_metrics['model_params'], 'model_size_mb': eval_metrics['model_size_mb'],
                'original_dataset_size_mb': orig_dataset_size_mb, 'optimized_dataset_size_mb': opt_dataset_size_mb,
                'dataset_reduction_pct': float((1 - opt_dataset_size_mb / orig_dataset_size_mb) * 100) if orig_dataset_size_mb > 0 else 0,
                'gpu_mem_peak_mb': eval_metrics['gpu_mem_peak_mb'],
                'convergence': fs_results.get('convergence', None),
                'best_fitness': fs_results.get('best_fitness', None),
            }
            
            master_results[method_name][fold_idx] = fold_result
            save_fold_result(fold_result, method_name, fold_idx)
            
            print(f"    L2 Test Acc: {eval_metrics['test_acc']*100:.2f}%, "
                  f"F1: {eval_metrics['test_f1_macro']*100:.2f}%, "
                  f"Features: {np.sum(final_mask)}/{TOTAL_FEATURES} "
                  f"({np.sum(final_mask)/TOTAL_FEATURES*100:.1f}%)")
            
            del model, train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader
            torch.cuda.empty_cache(); gc.collect()
    
    print(f"\n{'='*80}")
    print("ALL LAYER 2 EXPERIMENTS COMPLETED")
    print(f"{'='*80}")
    
    return master_results

print("Layer 2 experiment runner defined")


Layer 2 experiment runner defined


## Run Layer 2 Experiments

In [15]:
master_results = run_layer2_experiments()

STARTING LAYER 2 FEATURE SELECTION EXPERIMENTS

################################################################################
# FOLD 1/5
################################################################################
  Train: 704, Val: 175, Test: 286

  --- Layer 2: MUTUAL_INFO ---
    L1 combined features: 1256/2513
    Running Mutual Information (Layer 2)...
      Selected: 628/1256 (50.0%), Time: 6.2s
    L2 Test Acc: 98.95%, F1: 98.95%, Features: 628/2513 (25.0%)

  --- Layer 2: RFE ---
    L1 combined features: 1256/2513
    Running RFE (Layer 2)...
      Selected: 628/1256 (50.0%), Time: 1.2s
    L2 Test Acc: 97.90%, F1: 97.90%, Features: 628/2513 (25.0%)

  --- Layer 2: LASSO ---
    L1 combined features: 1256/2513
    Running LASSO (Layer 2)...
      Selected: 628/1256 (50.0%), Time: 0.2s
    L2 Test Acc: 91.96%, F1: 91.54%, Features: 628/2513 (25.0%)

  --- Layer 2: METAHEURISTICS (on best-meta combined L1) ---
    L1 combined best-meta features: 1285/2513
      depth: met

## Build Comprehensive Per-Fold Results CSV

In [16]:
rows = []

for method_name in ALL_METHODS_L2:
    for fold_idx in range(N_FOLDS):
        if fold_idx not in master_results[method_name]:
            continue
        r = master_results[method_name][fold_idx]
        
        row = {
            'Method': method_name,
            'Fold': fold_idx + 1,
            'Test Accuracy (%)': r['test_acc'] * 100,
            'Val Accuracy (%)': r['val_acc'] * 100,
            'F1 Macro (%)': r['test_f1_macro'] * 100,
            'F1 Weighted (%)': r['test_f1_weighted'] * 100,
            'Precision Macro (%)': r['test_precision_macro'] * 100,
            'Recall Macro (%)': r['test_recall_macro'] * 100,
            'Features Selected': r['num_features_selected'],
            'Features Total': r['num_features_total'],
            'Feature Retention (%)': r['feature_retention_pct'],
            'L1 Features': r.get('l1_features', TOTAL_FEATURES),
            'FS Time (s)': r['fs_execution_time'],
            'Train Time (s)': r['train_time_sec'],
            'Total Time (s)': r['total_time_sec'],
            'Model Params': r['model_params'],
            'Model Size (MB)': r['model_size_mb'],
            'Original Dataset (MB)': r['original_dataset_size_mb'],
            'Optimized Dataset (MB)': r['optimized_dataset_size_mb'],
            'Dataset Reduction (%)': r['dataset_reduction_pct'],
            'GPU Peak (MB)': r['gpu_mem_peak_mb'],
        }
        
        for mod_name in FEATURE_DIMS.keys():
            mod_ret = r['modality_retention'].get(mod_name, {})
            if isinstance(mod_ret, dict):
                row[f'{mod_name}_retention (%)'] = mod_ret.get('percentage', 0)
                row[f'{mod_name}_selected'] = mod_ret.get('selected', 0)
            else:
                row[f'{mod_name}_retention (%)'] = float(mod_ret)
        
        rows.append(row)

df_all = pd.DataFrame(rows)
df_all = df_all.round(4)

csv_path = RESULTS_ROOT / "all_results_per_fold.csv"
df_all.to_csv(csv_path, index=False)
print(f"Saved comprehensive per-fold CSV: {csv_path}")
print(f"Shape: {df_all.shape}")
print(df_all.head(20).to_string())


Saved comprehensive per-fold CSV: results_two_layer/all_results_per_fold.csv
Shape: (85, 27)
         Method  Fold  Test Accuracy (%)  Val Accuracy (%)  F1 Macro (%)  F1 Weighted (%)  Precision Macro (%)  Recall Macro (%)  Features Selected  Features Total  Feature Retention (%)  L1 Features  FS Time (s)  Train Time (s)  Total Time (s)  Model Params  Model Size (MB)  Original Dataset (MB)  Optimized Dataset (MB)  Dataset Reduction (%)  GPU Peak (MB)  depth_retention (%)  depth_selected  sensor_retention (%)  sensor_selected  skeleton_retention (%)  skeleton_selected
0   mutual_info     1            98.9510          100.0000       98.9495          98.9495              99.0260           98.9510                628            2513                24.9901         1256       6.2350          5.6954         11.9305        460566           1.7628                22.3362                  5.5818                75.0099        27.9590               0.9259               2               30.6748        

## Per-Method Summary Table

In [17]:
summary_rows = []
for method_name in ALL_METHODS_L2:
    method_df = df_all[df_all['Method'] == method_name]
    if len(method_df) == 0:
        continue
    
    row = {
        'Method': method_name,
        'Mean Test Acc (%)': method_df['Test Accuracy (%)'].mean(),
        'Std Test Acc (%)': method_df['Test Accuracy (%)'].std(),
        'Mean F1 Macro (%)': method_df['F1 Macro (%)'].mean(),
        'Std F1 Macro (%)': method_df['F1 Macro (%)'].std(),
        'Mean Features Selected': method_df['Features Selected'].mean(),
        'Mean Feature Retention (%)': method_df['Feature Retention (%)'].mean(),
        'Mean L1 Features': method_df['L1 Features'].mean(),
        'Mean FS Time (s)': method_df['FS Time (s)'].mean(),
        'Mean Train Time (s)': method_df['Train Time (s)'].mean(),
        'Mean Total Time (s)': method_df['Total Time (s)'].mean(),
        'Mean Model Params': method_df['Model Params'].mean(),
        'Mean Model Size (MB)': method_df['Model Size (MB)'].mean(),
        'Mean Dataset Reduction (%)': method_df['Dataset Reduction (%)'].mean(),
        'Mean GPU Peak (MB)': method_df['GPU Peak (MB)'].mean(),
    }
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).round(2)
df_summary.to_csv(RESULTS_ROOT / "summary_all_methods.csv", index=False)
print("\nSUMMARY TABLE (mean across folds):")
print(df_summary.to_string(index=False))



SUMMARY TABLE (mean across folds):
     Method  Mean Test Acc (%)  Std Test Acc (%)  Mean F1 Macro (%)  Std F1 Macro (%)  Mean Features Selected  Mean Feature Retention (%)  Mean L1 Features  Mean FS Time (s)  Mean Train Time (s)  Mean Total Time (s)  Mean Model Params  Mean Model Size (MB)  Mean Dataset Reduction (%)  Mean GPU Peak (MB)
mutual_info              95.39              4.31              95.09              4.73                   628.0                       24.99            1256.0              6.30                 6.12                12.42           460566.0                  1.76                       75.01               27.96
        rfe              96.51              2.99              96.42              3.10                   628.0                       24.99            1256.0              1.24                 6.13                 7.37           460566.0                  1.76                       75.01               27.96
      lasso              91.39              1.80 

## Comprehensive Visualizations

In [18]:
# ============================================================================
# PLOT 1: Test Accuracy per fold for all methods (heatmap)
# ============================================================================
pivot = df_all.pivot_table(values='Test Accuracy (%)', index='Method', columns='Fold')
pivot = pivot.reindex(ALL_METHODS_L2)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlGnBu', ax=ax, linewidths=0.5, cbar_kws={'label': 'Test Accuracy (%)'})
ax.set_title('Two-Layer FS: Test Accuracy (%) per Method per Fold', fontsize=16)
ax.set_ylabel('Method')
ax.set_xlabel('Fold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '01_accuracy_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 01_accuracy_heatmap.png")


Saved: 01_accuracy_heatmap.png


In [19]:
# ============================================================================
# PLOT 2: Box plot of test accuracy across folds
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 7))
methods_sorted = df_summary.sort_values('Mean Test Acc (%)', ascending=False)['Method'].tolist()

sns.boxplot(data=df_all, x='Method', y='Test Accuracy (%)', order=methods_sorted, ax=ax, palette='Set2')
sns.stripplot(data=df_all, x='Method', y='Test Accuracy (%)', order=methods_sorted, ax=ax, 
              color='black', alpha=0.5, size=4, jitter=True)
ax.set_title('Two-Layer FS: Test Accuracy Distribution Across Folds', fontsize=16)
ax.set_xlabel('')
plt.xticks(rotation=60, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '02_accuracy_boxplot.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 02_accuracy_boxplot.png")


Saved: 02_accuracy_boxplot.png


In [20]:
# ============================================================================
# PLOT 3: Feature Retention per method
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 7))
feat_summary = df_all.groupby('Method')['Feature Retention (%)'].agg(['mean', 'std']).reindex(ALL_METHODS_L2)
bars = ax.bar(feat_summary.index, feat_summary['mean'], yerr=feat_summary['std'], capsize=3, alpha=0.7, color='coral')
ax.axhline(y=100, color='r', linestyle='--', label='All Features (100%)')
ax.set_ylabel('Feature Retention (%)', fontsize=12)
ax.set_title('Two-Layer FS: Feature Retention by Method', fontsize=16)
ax.legend()
plt.xticks(rotation=60, ha='right')
ax.grid(axis='y', alpha=0.3)

for bar, mean_val in zip(bars, feat_summary['mean']):
    if not np.isnan(mean_val):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                f'{mean_val:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS_DIR / '03_feature_retention.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 03_feature_retention.png")


Saved: 03_feature_retention.png


In [21]:
# ============================================================================
# PLOT 4: Accuracy vs Feature Retention scatter
# ============================================================================
fig, ax = plt.subplots(figsize=(12, 8))

for method in ALL_METHODS_L2:
    mdf = df_all[df_all['Method'] == method]
    if len(mdf) == 0:
        continue
    ax.scatter(mdf['Feature Retention (%)'], mdf['Test Accuracy (%)'], 
               label=method, s=60, alpha=0.7)
    ax.scatter(mdf['Feature Retention (%)'].mean(), mdf['Test Accuracy (%)'].mean(),
               s=150, marker='X', edgecolors='black', linewidths=1.5)

ax.set_xlabel('Feature Retention (%)', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Two-Layer FS: Accuracy vs Feature Retention', fontsize=16)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '04_accuracy_vs_retention.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 04_accuracy_vs_retention.png")


Saved: 04_accuracy_vs_retention.png


In [22]:
# ============================================================================
# PLOT 5: Per-modality retention heatmap
# ============================================================================
modality_data = {}
for method in ALL_METHODS_L2:
    modality_data[method] = {}
    for mod in FEATURE_DIMS.keys():
        col = f'{mod}_retention (%)'
        if col in df_all.columns:
            mdf = df_all[df_all['Method'] == method]
            modality_data[method][mod] = mdf[col].mean() if len(mdf) > 0 else 0

mod_df = pd.DataFrame(modality_data).T
mod_df = mod_df.reindex(ALL_METHODS_L2)

fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(mod_df, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax, linewidths=0.5,
            vmin=0, vmax=100, cbar_kws={'label': 'Retention (%)'})
ax.set_title('Two-Layer FS: Average Modality-wise Feature Retention (%)', fontsize=14)
ax.set_ylabel('Method')
ax.set_xlabel('Modality')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '05_modality_retention_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 05_modality_retention_heatmap.png")


Saved: 05_modality_retention_heatmap.png


In [23]:
# ============================================================================
# PLOT 6: Execution time comparison
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

time_summary = df_all.groupby('Method')['FS Time (s)'].agg(['mean', 'std']).reindex(ALL_METHODS_L2)
bars = axes[0].bar(time_summary.index, time_summary['mean'], yerr=time_summary['std'], capsize=3, alpha=0.7, color='steelblue')
axes[0].set_ylabel('Feature Selection Time (s)', fontsize=12)
axes[0].set_title('Layer 2 Feature Selection Time', fontsize=14)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=60, ha='right')
axes[0].grid(axis='y', alpha=0.3)

total_summary = df_all.groupby('Method')['Total Time (s)'].agg(['mean', 'std']).reindex(ALL_METHODS_L2)
bars2 = axes[1].bar(total_summary.index, total_summary['mean'], yerr=total_summary['std'], capsize=3, alpha=0.7, color='darkorange')
axes[1].set_ylabel('Total Time (s)', fontsize=12)
axes[1].set_title('Total Time (FS + Training)', fontsize=14)
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=60, ha='right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '06_execution_time.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 06_execution_time.png")


Saved: 06_execution_time.png


In [24]:
# ============================================================================
# PLOT 7: Metaheuristics-only comparison
# ============================================================================
meta_method_names = [m for m in ALL_METHODS_L2 if m.startswith('meta_')]

if len(meta_method_names) > 0:
    meta_df = df_all[df_all['Method'].isin(meta_method_names)]
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 12))
    
    meta_acc = meta_df.groupby('Method')['Test Accuracy (%)'].agg(['mean', 'std']).reindex(meta_method_names)
    bars = axes[0].bar(range(len(meta_method_names)), meta_acc['mean'], yerr=meta_acc['std'], 
                        capsize=4, alpha=0.7, color=plt.cm.tab20(np.linspace(0, 1, len(meta_method_names))))
    axes[0].set_xticks(range(len(meta_method_names)))
    axes[0].set_xticklabels([m.replace('meta_', '') for m in meta_method_names], rotation=45, ha='right')
    axes[0].set_ylabel('Test Accuracy (%)')
    axes[0].set_title('Two-Layer FS Metaheuristics: Test Accuracy', fontsize=14)
    axes[0].grid(axis='y', alpha=0.3)
    
    for i, (bar, val) in enumerate(zip(bars, meta_acc['mean'])):
        if not np.isnan(val):
            axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + meta_acc['std'].iloc[i] + 0.5,
                        f'{val:.1f}', ha='center', va='bottom', fontsize=8)
    
    meta_feat = meta_df.groupby('Method')['Feature Retention (%)'].agg(['mean', 'std']).reindex(meta_method_names)
    bars2 = axes[1].bar(range(len(meta_method_names)), meta_feat['mean'], yerr=meta_feat['std'],
                         capsize=4, alpha=0.7, color=plt.cm.tab20(np.linspace(0, 1, len(meta_method_names))))
    axes[1].set_xticks(range(len(meta_method_names)))
    axes[1].set_xticklabels([m.replace('meta_', '') for m in meta_method_names], rotation=45, ha='right')
    axes[1].set_ylabel('Feature Retention (%)')
    axes[1].set_title('Two-Layer FS Metaheuristics: Feature Retention', fontsize=14)
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '07_metaheuristics_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Saved: 07_metaheuristics_comparison.png")


Saved: 07_metaheuristics_comparison.png


In [25]:
# ============================================================================
# PLOT 8: Convergence curves for metaheuristics
# ============================================================================
meta_method_names = [m for m in ALL_METHODS_L2 if m.startswith('meta_')]

if len(meta_method_names) > 0:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    for method in meta_method_names:
        all_conv = []
        for fold_idx in range(N_FOLDS):
            if fold_idx in master_results[method]:
                conv = master_results[method][fold_idx].get('convergence', None)
                if conv is not None:
                    all_conv.append(conv)
        
        if all_conv:
            max_len = max(len(c) for c in all_conv)
            padded = []
            for c in all_conv:
                if len(c) < max_len:
                    c = list(c) + [c[-1]] * (max_len - len(c))
                padded.append(c)
            
            avg_conv = np.mean(padded, axis=0)
            ax.plot(avg_conv, label=method.replace('meta_', ''), linewidth=1.5)
    
    ax.set_xlabel('Iteration', fontsize=12)
    ax.set_ylabel('Fitness (1 - Accuracy)', fontsize=12)
    ax.set_title('Layer 2 Metaheuristic Convergence Curves (averaged)', fontsize=14)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '08_convergence_curves.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Saved: 08_convergence_curves.png")


Saved: 08_convergence_curves.png


In [26]:
# ============================================================================
# PLOT 9: Standard vs Metaheuristics grouped comparison
# ============================================================================
standard_methods_plot = ['mutual_info', 'rfe', 'lasso']
meta_methods_plot = [m for m in ALL_METHODS_L2 if m.startswith('meta_')]

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

all_std = df_all[df_all['Method'].isin(standard_methods_plot)]['Test Accuracy (%)']
all_meta = df_all[df_all['Method'].isin(meta_methods_plot)]['Test Accuracy (%)']
bp = axes[0].boxplot([all_std, all_meta], labels=['Standard\n(MI, RFE, LASSO)', 'Metaheuristics\n(14 optimizers)'],
                      patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightsalmon')
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Standard vs Metaheuristic Accuracy', fontsize=13)
axes[0].grid(axis='y', alpha=0.3)

all_std_f = df_all[(df_all['Method'].isin(standard_methods_plot)) & (df_all['Method'] != 'baseline')]['Feature Retention (%)']
all_meta_f = df_all[df_all['Method'].isin(meta_methods_plot)]['Feature Retention (%)']
bp2 = axes[1].boxplot([all_std_f, all_meta_f], labels=['Standard', 'Metaheuristics'],
                       patch_artist=True)
bp2['boxes'][0].set_facecolor('lightblue')
bp2['boxes'][1].set_facecolor('lightsalmon')
axes[1].set_ylabel('Feature Retention (%)')
axes[1].set_title('Standard vs Metaheuristic Feature Retention', fontsize=13)
axes[1].grid(axis='y', alpha=0.3)

all_std_t = df_all[(df_all['Method'].isin(standard_methods_plot)) & (df_all['Method'] != 'baseline')]['Total Time (s)']
all_meta_t = df_all[df_all['Method'].isin(meta_methods_plot)]['Total Time (s)']
bp3 = axes[2].boxplot([all_std_t, all_meta_t], labels=['Standard', 'Metaheuristics'],
                       patch_artist=True)
bp3['boxes'][0].set_facecolor('lightblue')
bp3['boxes'][1].set_facecolor('lightsalmon')
axes[2].set_ylabel('Total Time (s)')
axes[2].set_title('Standard vs Metaheuristic Time', fontsize=13)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '09_standard_vs_metaheuristics.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 09_standard_vs_metaheuristics.png")


Saved: 09_standard_vs_metaheuristics.png


In [27]:
# ============================================================================
# PLOT 10: Per-fold accuracy line plot
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 8))

for method in ALL_METHODS_L2:
    mdf = df_all[df_all['Method'] == method].sort_values('Fold')
    if len(mdf) > 0:
        linewidth = 2.5 if method in ['mutual_info', 'rfe', 'lasso'] else 1.0
        alpha = 1.0 if method in ['mutual_info', 'rfe', 'lasso'] else 0.6
        ax.plot(mdf['Fold'], mdf['Test Accuracy (%)'], marker='o', markersize=4,
                label=method, linewidth=linewidth, alpha=alpha)

ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Two-Layer FS: Test Accuracy Across Folds', fontsize=16)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(alpha=0.3)
ax.set_xticks(range(1, N_FOLDS + 1))
plt.tight_layout()
plt.savefig(PLOTS_DIR / '10_per_fold_accuracy_lines.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 10_per_fold_accuracy_lines.png")


Saved: 10_per_fold_accuracy_lines.png


## Best Metaheuristic Per Fold (Layer 2)

In [28]:
meta_methods_l2 = [m for m in ALL_METHODS_L2 if m.startswith('meta_')]
modality_names = list(FEATURE_DIMS.keys())

print("=" * 90)
print("BEST METAHEURISTIC PER FOLD - LAYER 2 (selected by highest test accuracy)")
print("=" * 90)

best_per_fold = []

for fold_idx in range(N_FOLDS):
    best_method = None
    best_test_acc = -1.0
    best_result = None

    for method_name in meta_methods_l2:
        if fold_idx in master_results[method_name]:
            r = master_results[method_name][fold_idx]
            if r['test_acc'] > best_test_acc:
                best_test_acc = r['test_acc']
                best_method = method_name
                best_result = r

    if best_result is None:
        print(f"  Fold {fold_idx + 1}: No metaheuristic results found!")
        continue

    mod_ret = best_result['modality_retention']
    mod_retention_pct = {}
    for mod in modality_names:
        if mod in mod_ret and isinstance(mod_ret[mod], dict):
            mod_retention_pct[mod] = mod_ret[mod].get('percentage', 0.0)
        else:
            mod_retention_pct[mod] = float(mod_ret.get(mod, 0.0))

    fold_entry = {
        'Fold': fold_idx + 1,
        'Best Metaheuristic': best_method,
        'Test Accuracy (%)': best_result['test_acc'] * 100,
        'Feature Retention (%)': best_result['feature_retention_pct'],
        'Features Selected': best_result['num_features_selected'],
        'Features Total': best_result['num_features_total'],
    }
    for mod in modality_names:
        fold_entry[f'{mod} Retention (%)'] = mod_retention_pct[mod]

    best_per_fold.append(fold_entry)

    print(f"\n  Fold {fold_idx + 1}: Best = {best_method}")
    print(f"    Test Accuracy:      {fold_entry['Test Accuracy (%)']:.2f}%")
    print(f"    Feature Retention:  {fold_entry['Feature Retention (%)']:.2f}% "
          f"({fold_entry['Features Selected']}/{fold_entry['Features Total']})")
    mod_str = ', '.join([f"{mod}={mod_retention_pct[mod]:.1f}%" for mod in modality_names])
    print(f"    Modality Retention: {mod_str}")

df_best_meta = pd.DataFrame(best_per_fold)

print("\n" + "=" * 90)
print("AVERAGE ACROSS ALL FOLDS (Best Metaheuristic per Fold - Layer 2)")
print("=" * 90)

if len(df_best_meta) > 0:
    avg_test_acc = df_best_meta['Test Accuracy (%)'].mean()
    std_test_acc = df_best_meta['Test Accuracy (%)'].std()
    avg_feat_ret = df_best_meta['Feature Retention (%)'].mean()
    std_feat_ret = df_best_meta['Feature Retention (%)'].std()

    print(f"\n  Average Test Accuracy:     {avg_test_acc:.2f}% ± {std_test_acc:.2f}%")
    print(f"  Average Feature Retention: {avg_feat_ret:.2f}% ± {std_feat_ret:.2f}%")

    print(f"\n  Average Modality-wise Feature Retention:")
    for mod in modality_names:
        col = f'{mod} Retention (%)'
        if col in df_best_meta.columns:
            avg_mod = df_best_meta[col].mean()
            std_mod = df_best_meta[col].std()
            print(f"    {mod:>10s}: {avg_mod:.2f}% ± {std_mod:.2f}%")

    csv_path = RESULTS_ROOT / "best_metaheuristic_per_fold.csv"
    df_best_meta.to_csv(csv_path, index=False)
    print(f"\nSaved to: {csv_path}")


BEST METAHEURISTIC PER FOLD - LAYER 2 (selected by highest test accuracy)

  Fold 1: Best = meta_SCA
    Test Accuracy:      95.10%
    Feature Retention:  7.36% (185/2513)
    Modality Retention: depth=8.8%, sensor=6.0%, skeleton=7.7%

  Fold 2: Best = meta_CS
    Test Accuracy:      92.53%
    Feature Retention:  22.60% (568/2513)
    Modality Retention: depth=25.5%, sensor=20.2%, skeleton=23.2%

  Fold 3: Best = meta_DE
    Test Accuracy:      98.64%
    Feature Retention:  22.01% (553/2513)
    Modality Retention: depth=26.4%, sensor=21.8%, skeleton=21.5%

  Fold 4: Best = meta_MVO
    Test Accuracy:      94.55%
    Feature Retention:  24.83% (624/2513)
    Modality Retention: depth=25.9%, sensor=26.1%, skeleton=24.2%

  Fold 5: Best = meta_JAYA
    Test Accuracy:      97.47%
    Feature Retention:  23.32% (586/2513)
    Modality Retention: depth=13.9%, sensor=25.9%, skeleton=23.5%

AVERAGE ACROSS ALL FOLDS (Best Metaheuristic per Fold - Layer 2)

  Average Test Accuracy:     95.66